# Vecka 2 – Kapitel 2: kod

Koduppgifterna 8–12.

## Uppgift 8 – spara och ladda en modell med joblib

In [1]:
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from joblib import dump, load

X, y = make_regression(n_samples=20000, n_features=3, noise=0.1, random_state=42)  # syntetisk data

model = LinearRegression().fit(X, y)   # tränar
dump(model, "linear_model.joblib")     # sparar till fil
model_loaded = load("linear_model.joblib")  # laddar tillbaka

print(model_loaded.predict(X[:5]))     # predikterar med laddad modell

[ 105.19825923 -124.46546207  -15.07418334  103.66450947   92.8938913 ]


## Uppgift 9 – fullständigt flöde på data_01.csv (test 20 %, validering 15 % av resten)

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

# a) Läs in
df = pd.read_csv("../dataset/data_01.csv")

# b) X och y
X = df.drop(columns=["target"])
y = df["target"]

# c) Träning / validering / test
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.15, random_state=42)

# d) Träna två modeller
modeller = {"Linjär regression": LinearRegression(),
            "Beslutsträd":       DecisionTreeRegressor(random_state=42)}
for m in modeller.values():
    m.fit(X_train, y_train)

# e) Utvärdera på valideringsdatan
rmse = {namn: mean_squared_error(y_val, m.predict(X_val)) ** 0.5
        for namn, m in modeller.items()}
for namn, v in rmse.items():
    print(f"{namn:20} val-RMSE: {v:.3f}")

# f) Träna om bästa modellen på träning + validering
best = min(rmse, key=rmse.get)
modell = modeller[best].fit(
    pd.concat([X_train, X_val]), pd.concat([y_train, y_val]))

# g) Utvärdera på testdatan
print(f"\nBäst: {best}  test-RMSE: "
      f"{mean_squared_error(y_test, modell.predict(X_test)) ** 0.5:.3f}")

# h) Träna om på hela datasetet
modell.fit(X, y)
print("Omtränad på hela datan: OK")

Linjär regression    val-RMSE: 3.592
Beslutsträd          val-RMSE: 99.479

Bäst: Linjär regression  test-RMSE: 3.372
Omtränad på hela datan: OK


## Uppgift 10 – salary_dataset.csv med k-delad korsvalidering (inget valideringsset)

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

# a) Läs in. y = Salary, x = YearsExperience
df = pd.read_csv("../dataset/salary_dataset.csv")
X = df[["YearsExperience"]]
y = df["Salary"]

# b) Träning / test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# c) Två modeller med korsvalidering (cv=5)
modeller = {"Linjär regression": LinearRegression(),
            "Beslutsträd":       DecisionTreeRegressor(random_state=42)}
cv_rmse = {}
for namn, m in modeller.items():
    cv = cross_validate(m, X_train, y_train,
                        scoring="neg_root_mean_squared_error", cv=5)
    cv_rmse[namn] = -cv["test_score"].mean()
    print(f"{namn:20} CV-RMSE: {cv_rmse[namn]:.0f}")

# d) Utvärdera bästa modellen på testsetet
best = min(cv_rmse, key=cv_rmse.get)
modell = modeller[best].fit(X_train, y_train)
print(f"\nBäst: {best}  test-RMSE: "
      f"{mean_squared_error(y_test, modell.predict(X_test)) ** 0.5:.0f}")

Linjär regression    CV-RMSE: 5293
Beslutsträd          CV-RMSE: 5612

Bäst: Linjär regression  test-RMSE: 7059


## Uppgift 11 – kategorisk data på mpg (droppa name, one-hot på origin)

Obs: seaborn hämtar `mpg` från nätet första gången.

In [4]:
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# a) Läs in
df = sns.load_dataset("mpg")

# b) Droppa rader med saknade värden
df = df.dropna()

# c) Droppa name
df = df.drop(columns=["name"])

# d) Dummy-encoda origin (drop_first=True -> 2 kolumner)
df = pd.get_dummies(df, columns=["origin"], drop_first=True)

# e) X och y (mpg = beroende variabel)
X = df.drop(columns=["mpg"])
y = df["mpg"]

# f) Träning / test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# g) Träna linjär regression och utvärdera
modell = LinearRegression().fit(X_train, y_train)
rmse = mean_squared_error(y_test, modell.predict(X_test)) ** 0.5
print(f"test-RMSE: {rmse:.2f}")

test-RMSE: 3.26


## Uppgift 12 – förbättra RandomForest på huspris-exemplet

Kräver hela kodexemplet från avsnitt 2.2 (X_train/y_train därifrån). Här visas trimnings-metoden.

In [5]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

param = {"n_estimators": [100, 300],
         "max_features": [4, 6, 8],
         "max_depth": [None, 10, 20]}

sok = GridSearchCV(RandomForestRegressor(random_state=42), param,
                   scoring="neg_root_mean_squared_error", cv=5, n_jobs=-1)
# sok.fit(X_train, y_train)   # <- använd X_train/y_train från huspris-exemplet
# print(sok.best_params_, -sok.best_score_)